In [63]:
import json
import pandas as pd
import duckdb
import glob
from pathlib import Path

# Lendo dados

In [64]:
caminhos_arquivos = glob.glob('dados_brutos/json/*.json')

In [65]:
# 1. LISTAS PARA ACUMULAR OS DADOS
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

print("Iniciando o processamento dos arquivos JSON...")
#caminhos_arquivos = glob.glob('dados_brutos/*.json')

if not caminhos_arquivos:
    print("ERRO: Nenhum arquivo JSON encontrado na pasta 'dados_brutos/'.")
    exit()

# ---------------------------------------------------------
# 2. EXTRAÇÃO E ACHATAMENTO (FLATTEN)
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)
        
        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            continue
            
        # --- PESSOAS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)
        
        # --- BANCAS ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria 
                    if 'membros_banca' in df_temp.columns:
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel   
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA ---
        prod_bib = dados.get('producao_bibliografica', {})
        def add_to_list(chave, lista_destino):
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA ---
        prod_tec = dados.get('producao_tecnica', {})
        def add_to_list_tec(chave, lista_destino):
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES ---
        patentes = dados.get('patentes_registros', {})
        def add_to_list_pat(chave, lista_destino):
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)


# ---------------------------------------------------------
# 3. CONSOLIDAÇÃO EM DATAFRAMES EXPLÍCITOS
# ---------------------------------------------------------
print("Consolidando DataFrames...")

def consolidar(lista):
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames Patentes
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

Iniciando o processamento dos arquivos JSON...
Consolidando DataFrames...


# Tratando dados

## Informações pessoais

In [66]:
df_pessoas.info()

<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id_lattes              39 non-null     str  
 1   nome_completo          39 non-null     str  
 2   nome_citacoes          39 non-null     str  
 3   sexo                   39 non-null     str  
 4   rotulo                 39 non-null     str  
 5   periodo                39 non-null     str  
 6   bolsa_produtividade    39 non-null     str  
 7   endereco_profissional  39 non-null     str  
 8   atualizacao_cv         39 non-null     str  
 9   url                    39 non-null     str  
 10  texto_resumo           39 non-null     str  
dtypes: str(11)
memory usage: 3.5 KB


In [67]:
# 1. Substituir strings vazias e espaços em branco por NaN
# Usa expressão regular para pegar "" ou "   "
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/03/2026,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",05/12/2024,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
2,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",12/03/2026,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
3,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/07/2025,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...
4,0523104569378276,Marcia Helena Costa Fampa,"FAMPA, M. H. C.;FAMPA, M.;FAMPA, MARCIA H.C.;F...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade Federal do Rio de Janeiro, Progra...",07/04/2026,http://lattes.cnpq.br/0523104569378276,Marcia é Professora da Universidade Federal do...
5,2002515486942024,Jayme Luiz Szwarcfiter,"SZWARCFITER, J. L.;Szwarcfiter, Jayme L.;Jayme...",Masculino,* Sem rótulo,NaN,Nível 1A (***,"Universidade Federal do Rio de Janeiro, COPPE ...",05/09/2025,http://lattes.cnpq.br/2002515486942024,Possui graduação em Engenharia Eletrônica pela...
6,9358511568098561,Edmundo Albuquerque de Souza e Silva,"de Souza e Silva, E.;de Souza e Silva, Edmundo...",Masculino,* Sem rótulo,NaN,Nível SR,"Universidade Federal do Rio de Janeiro, Instit...",15/01/2025,http://lattes.cnpq.br/9358511568098561,Edmundo de Souza e Silva é Engenheiro Elétrico...
7,5815607228657970,Henrique Luiz Cukierman,"CUKIERMAN, H. L.;CUKIERMAN, HENRIQUE LUIZ;CUKI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",08/11/2025,http://lattes.cnpq.br/5815607228657970,Possui graduação em Engenharia de Sistemas pel...
8,2704717555047499,Priscila Machado Vieira Lima,"LIMA, P. M. V.;LIMA, PRISCILA M. V.;LIMA, PRIS...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Núcleo...",08/08/2025,http://lattes.cnpq.br/2704717555047499,Possui graduação em Informática pela Universid...
9,5349830056087028,Pedro Henrique González Silva,"González, P.H.;González, Pedro Henrique;GONZAL...",Masculino,* Sem rótulo,NaN,Nível C,"Universidade Federal do Rio de Janeiro, PESC -...",29/04/2026,http://lattes.cnpq.br/5349830056087028,Professor Adjunto na Universidade Federal do R...


In [68]:
# 2. Tratamento da Data de Atualização
# Converte a string '15/10/2025' para um tipo datetime
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'], 
        format='%d/%m/%Y', 
        errors='coerce' # Se tiver uma data bizarra (ex: 99/99/9999), vira nulo em vez de quebrar o script
    )

In [69]:
# 3. Limpeza do campo Rótulo
if 'rotulo' in df_pessoas.columns:
    # Remove o asterisco e espaços em branco nas pontas
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    # Se o rótulo ficou "Sem rótulo", transforma em nulo verdadeiro
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

In [70]:
# 5. Garantia de Tipagem da Chave Primária e Textos Longos
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

In [71]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [72]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [73]:
df_pessoas.head()

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2026-03-30,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2024-12-05,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
2,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,NaN,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",2026-03-12,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
3,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2025-07-30,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...
4,0523104569378276,Marcia Helena Costa Fampa,"FAMPA, M. H. C.;FAMPA, M.;FAMPA, MARCIA H.C.;F...",Masculino,NaN,NaN,Nível 2,"Universidade Federal do Rio de Janeiro, Progra...",2026-04-07,http://lattes.cnpq.br/0523104569378276,Marcia é Professora da Universidade Federal do...


## Informações acerca de periódicos publicados

In [74]:
df_bib_artigos.head()

,titulo,ano,autores,revista,volume,numero,paginas,issn,doi,qualis,id_lattes
0,On the (In)Dependence of the Peano Axioms for ...,2021,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",History and Philosophy of Logic,?,,1-19,1464-5149,http://dx.doi.org/10.1080/01445340.2021.1971005,,0211300683784278
1,Short proofs on the structure of general parti...,2021,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",DISCRETE APPLIED MATHEMATICS,303,,8-13,0166-218X,http://dx.doi.org/10.1016/j.dam.2020.09.007,,0211300683784278
2,Transversals of longest paths,2020,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",DISCRETE MATHEMATICS,343,,111717,0012-365X,http://dx.doi.org/10.1016/j.disc.2019.111717,,0211300683784278
3,Intersection of longest paths in graph classes,2020,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",DISCRETE APPLIED MATHEMATICS,281,,96-105,0166-218X,http://dx.doi.org/10.1016/j.dam.2019.03.022,,0211300683784278
4,On Edge-magic Labelings of Forests,2019,"CERIOLI, M. R.; FERNANDES, C. G. ; LEE, O. ; L...",ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,346,,299-307,1571-0661,http://dx.doi.org/10.1016/j.entcs.2019.08.027,,0211300683784278


In [76]:
print("Aplicando tratamentos na tabela 'bib_artigos'...")

if not df_bib_artigos.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da Revista (Periódico)
    if 'revista' in df_bib_artigos.columns:
        # Força maiúsculo e remove espaços extras no início e no fim
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 4. Tratamento do Ano (Garantir que seja número inteiro)
    if 'ano' in df_bib_artigos.columns:
        # errors='coerce' transforma erros (ex: "Sem ano") em NaN
        # Int64 é o tipo inteiro do Pandas que aceita valores nulos
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 5. Tratamento de Título, DOI e ISSN (Apenas remover espaços ocultos)
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 6. Garantir tipagem da chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento da tabela 'bib_artigos' concluído!")
display(df_bib_artigos[['ano', 'revista', 'doi']].head())

Aplicando tratamentos na tabela 'bib_artigos'...
Tratamento da tabela 'bib_artigos' concluído!


,ano,revista,doi
0,2021,HISTORY AND PHILOSOPHY OF LOGIC,http://dx.doi.org/10.1080/01445340.2021.1971005
1,2021,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2020.09.007
2,2020,DISCRETE MATHEMATICS,http://dx.doi.org/10.1016/j.disc.2019.111717
3,2020,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2019.03.022
4,2019,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,http://dx.doi.org/10.1016/j.entcs.2019.08.027


In [78]:
df_bib_artigos.info()

<class 'pandas.DataFrame'>
RangeIndex: 1997 entries, 0 to 1996
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     1997 non-null   str  
 1   ano        1997 non-null   Int64
 2   autores    1997 non-null   str  
 3   revista    1997 non-null   str  
 4   volume     1960 non-null   str  
 5   numero     220 non-null    str  
 6   paginas    1980 non-null   str  
 7   issn       1956 non-null   str  
 8   doi        1516 non-null   str  
 9   qualis     0 non-null      str  
 10  id_lattes  1997 non-null   str  
dtypes: Int64(1), str(10)
memory usage: 173.7 KB


In [ ]:
df_bib_artigos['qtd_autores'].unique()

<IntegerArray>
[4, 2, 5, 6, 3, 1, 8, 7, 9, 10, 15, 12, 13, 21, 18, 19, 16, 17, 14, 11, 20]
Length: 21, dtype: Int64

## Conectando com DuckDB

In [79]:
import duckdb

print("Salvando as tabelas no DuckDB...")
# Conecta ao arquivo (ele será criado se não existir)
con = duckdb.connect('pesquisadores.duckdb')

# Salva o DataFrame 'df_pessoas' numa tabela SQL chamada 'tb_pessoas'
con.execute("CREATE OR REPLACE TABLE tb_pessoas AS SELECT * FROM df_pessoas")

# Salva o DataFrame 'df_bib_artigos' numa tabela SQL chamada 'tb_artigos'
con.execute("CREATE OR REPLACE TABLE tb_artigos AS SELECT * FROM df_bib_artigos")

con.close()
print("Processo finalizado! Banco 'pesquisadores.duckdb' atualizado e salvo.")

Salvando as tabelas no DuckDB...
Processo finalizado! Banco 'pesquisadores.duckdb' atualizado e salvo.
